# Phase 0: Setup Environment & Data Preparation

**Muc tieu:** Cai dat moi truong, download datasets, xay dung Knowledge Graph san sang cho cac phase sau.

**Datasets:**
- WebQSP (QA tren Freebase KG)
- HotpotQA (Multi-hop QA)
- ConceptNet (da co tu KnowGPT)

**Output:** Data da xu ly, KG dang graph structure luu tren Drive

## 0.1 Mount Drive & Cau hinh duong dan

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

# === CAU HINH DUONG DAN CHINH ===
# Thay doi ROOT_DIR cho phu hop voi Drive cua ban
ROOT_DIR = '/content/drive/MyDrive/ThS/Research/colab/graph_rag'
DATA_DIR = os.path.join(ROOT_DIR, 'data')
KG_DIR = os.path.join(DATA_DIR, 'knowledge_graphs')
DATASET_DIR = os.path.join(DATA_DIR, 'datasets')
CHECKPOINT_DIR = os.path.join(ROOT_DIR, 'checkpoints')
RESULTS_DIR = os.path.join(ROOT_DIR, 'results')

for d in [ROOT_DIR, DATA_DIR, KG_DIR, DATASET_DIR, CHECKPOINT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'ROOT_DIR: {ROOT_DIR}')
print('Tat ca thu muc da san sang.')

Mounted at /content/drive
ROOT_DIR: /content/drive/MyDrive/ThS/Research/research/COLAB/RUNNING
Tat ca thu muc da san sang.


## 0.2 Cai dat thu vien

In [2]:
%%capture
!pip install torch-geometric
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-$(python -c 'import torch; print(torch.__version__)').html
!pip install transformers sentence-transformers
!pip install networkx igraph
!pip install datasets jsonlines
!pip install openai
!pip install wandb
!pip install pyvis

In [3]:
# Verify installations
import torch
import torch_geometric
import networkx as nx
import transformers

print(f'PyTorch: {torch.__version__}')
print(f'PyG: {torch_geometric.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'NetworkX: {nx.__version__}')
print(f'Transformers: {transformers.__version__}')

PyTorch: 2.10.0+cu128
PyG: 2.7.0
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
NetworkX: 3.6.1
Transformers: 5.0.0


## 0.3 Download Datasets

### 0.3.1 WebQSP
WebQuestions Semantic Parses - QA tren Freebase Knowledge Graph.
- Train: 3,098 questions
- Test: 1,639 questions
- Moi question co SPARQL query va answer entities tren Freebase

In [4]:
WEBQSP_DIR = os.path.join(DATASET_DIR, 'webqsp')
os.makedirs(WEBQSP_DIR, exist_ok=True)

# Download WebQSP
# Option 1: Tu repo GNN-RAG (da format san)
if not os.path.exists(os.path.join(WEBQSP_DIR, 'train.json')):
    !git clone https://github.com/cmavro/GNN-RAG.git /tmp/gnn_rag_repo
    !cp -r /tmp/gnn_rag_repo/data/webqsp/* {WEBQSP_DIR}/
    print('WebQSP downloaded tu GNN-RAG repo')
else:
    print('WebQSP da ton tai')

# Kiem tra
for f in os.listdir(WEBQSP_DIR):
    fpath = os.path.join(WEBQSP_DIR, f)
    size = os.path.getsize(fpath) / 1024 / 1024
    print(f'  {f}: {size:.1f} MB')

Cloning into '/tmp/gnn_rag_repo'...
remote: Enumerating objects: 241, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 241 (delta 32), reused 18 (delta 18), pack-reused 169 (from 1)
Receiving objects: 100% (241/241), 17.92 MiB | 9.18 MiB/s, done.
Resolving deltas: 100% (67/67), done.
cp: cannot stat '/tmp/gnn_rag_repo/data/webqsp/*': No such file or directory
WebQSP downloaded tu GNN-RAG repo


### 0.3.2 HotpotQA
Multi-hop QA - can truy xuat nhieu documents de tra loi.
- Train: 90,564 questions
- Dev: 7,405 questions
- Distractor setting: 10 paragraphs, 2 gold

In [5]:
HOTPOTQA_DIR = os.path.join(DATASET_DIR, 'hotpotqa')
os.makedirs(HOTPOTQA_DIR, exist_ok=True)

if not os.path.exists(os.path.join(HOTPOTQA_DIR, 'hotpot_dev_distractor_v1.json')):
    !wget -q http://curtis.ml.cmu.edu/datasets/hotpot/hotpot_train_v1.1.json -O {HOTPOTQA_DIR}/hotpot_train_v1.1.json
    !wget -q http://curtis.ml.cmu.edu/datasets/hotpot/hotpot_dev_distractor_v1.json -O {HOTPOTQA_DIR}/hotpot_dev_distractor_v1.json
    print('HotpotQA downloaded')
else:
    print('HotpotQA da ton tai')

import json
with open(os.path.join(HOTPOTQA_DIR, 'hotpot_dev_distractor_v1.json')) as f:
    hotpot_dev = json.load(f)
print(f'HotpotQA dev: {len(hotpot_dev)} questions')
print(f'Sample: {hotpot_dev[0]["question"]}')
print(f'Answer: {hotpot_dev[0]["answer"]}')
print(f'Type: {hotpot_dev[0]["type"]}, Level: {hotpot_dev[0]["level"]}')

HotpotQA downloaded
HotpotQA dev: 7405 questions
Sample: Were Scott Derrickson and Ed Wood of the same nationality?
Answer: yes
Type: comparison, Level: hard


### 0.3.3 Freebase Subgraph (cho WebQSP)
Freebase subgraph chua cac entities va relations lien quan den WebQSP.

In [6]:
FREEBASE_DIR = os.path.join(KG_DIR, 'freebase')
os.makedirs(FREEBASE_DIR, exist_ok=True)

# Download Freebase subgraph tu GNN-RAG repo
# Luu y: file co the lon, can High-RAM runtime
if not os.path.exists(os.path.join(FREEBASE_DIR, 'entities.txt')):
    print('Downloading Freebase subgraph...')
    print('NOTE: Kiem tra repo GNN-RAG de lay link download chinh xac')
    print('Co the can download thu cong va upload len Drive')
    # !cp -r /tmp/gnn_rag_repo/data/freebase/* {FREEBASE_DIR}/
else:
    print('Freebase subgraph da ton tai')

NOTE: Kiem tra repo GNN-RAG de lay link download chinh xac
Co the can download thu cong va upload len Drive


## 0.4 Xay dung Graph Structure

Chuyen KG tu raw triples -> NetworkX graph -> PyG Data object

In [7]:
import networkx as nx
import pickle
import numpy as np
from collections import defaultdict


class KnowledgeGraph:
    """Knowledge Graph wrapper cho research.

    Ho tro:
    - Load tu triples (head, relation, tail)
    - Chuyen sang NetworkX graph
    - Chuyen sang PyG Data
    - Truy van neighbors, subgraph extraction
    """

    def __init__(self):
        self.graph = nx.MultiDiGraph()
        self.entity2id = {}
        self.id2entity = {}
        self.relation2id = {}
        self.id2relation = {}
        self.triples = []  # list of (h, r, t)

    def load_triples(self, triples_list):
        """Load tu list of (head, relation, tail) strings."""
        for h, r, t in triples_list:
            # Entity mapping
            for e in [h, t]:
                if e not in self.entity2id:
                    idx = len(self.entity2id)
                    self.entity2id[e] = idx
                    self.id2entity[idx] = e
            # Relation mapping
            if r not in self.relation2id:
                idx = len(self.relation2id)
                self.relation2id[r] = idx
                self.id2relation[idx] = r

            self.triples.append((h, r, t))
            self.graph.add_edge(h, t, relation=r)

        print(f'Loaded KG: {len(self.entity2id)} entities, '
              f'{len(self.relation2id)} relations, {len(self.triples)} triples')

    def load_from_file(self, filepath, sep='\t'):
        """Load tu file TSV/CSV: head<sep>relation<sep>tail"""
        triples = []
        with open(filepath) as f:
            for line in f:
                parts = line.strip().split(sep)
                if len(parts) >= 3:
                    triples.append((parts[0], parts[1], parts[2]))
        self.load_triples(triples)

    def get_neighbors(self, entity, max_hops=1):
        """Lay neighbors trong k-hop."""
        if entity not in self.graph:
            return set()
        if max_hops == 1:
            return set(self.graph.successors(entity)) | set(self.graph.predecessors(entity))
        # BFS k-hop
        visited = {entity}
        frontier = {entity}
        for _ in range(max_hops):
            next_frontier = set()
            for node in frontier:
                next_frontier |= set(self.graph.successors(node))
                next_frontier |= set(self.graph.predecessors(node))
            next_frontier -= visited
            visited |= next_frontier
            frontier = next_frontier
        visited.discard(entity)
        return visited

    def extract_subgraph(self, seed_entities, max_hops=2, max_nodes=500):
        """Trich xuat subgraph tu seed entities."""
        nodes = set(seed_entities)
        frontier = set(seed_entities)
        for _ in range(max_hops):
            if len(nodes) >= max_nodes:
                break
            next_frontier = set()
            for node in frontier:
                if node in self.graph:
                    next_frontier |= set(self.graph.successors(node))
                    next_frontier |= set(self.graph.predecessors(node))
            next_frontier -= nodes
            nodes |= next_frontier
            frontier = next_frontier
            if len(nodes) >= max_nodes:
                nodes = set(list(nodes)[:max_nodes])
                break
        return self.graph.subgraph(nodes).copy()

    def shortest_path(self, source, target):
        """Tim shortest path giua 2 entities."""
        try:
            path = nx.shortest_path(self.graph, source, target)
            return path
        except nx.NetworkXNoPath:
            return None

    def to_pyg_data(self):
        """Chuyen sang PyTorch Geometric Data object."""
        from torch_geometric.data import Data

        edge_index = []
        edge_type = []
        for h, r, t in self.triples:
            edge_index.append([self.entity2id[h], self.entity2id[t]])
            edge_type.append(self.relation2id[r])

        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_type = torch.tensor(edge_type, dtype=torch.long)
        num_nodes = len(self.entity2id)

        data = Data(edge_index=edge_index, edge_type=edge_type, num_nodes=num_nodes)
        return data

    def save(self, filepath):
        """Luu KG object."""
        with open(filepath, 'wb') as f:
            pickle.dump(self, f)
        print(f'Saved KG to {filepath}')

    @staticmethod
    def load(filepath):
        """Load KG object."""
        with open(filepath, 'rb') as f:
            kg = pickle.load(f)
        print(f'Loaded KG: {len(kg.entity2id)} entities, {len(kg.relation2id)} relations')
        return kg


print('KnowledgeGraph class defined.')

KnowledgeGraph class defined.


### Test voi ConceptNet (da co tu KnowGPT)

In [8]:
# === Test voi ConceptNet ===
# Thay duong dan bang duong dan ConceptNet cua ban tren Drive
CONCEPTNET_PATH = '/content/drive/MyDrive/ABC/KnowGPT/conceptnet_en.csv'

if os.path.exists(CONCEPTNET_PATH):
    import pandas as pd
    df = pd.read_csv(CONCEPTNET_PATH, sep='\t', header=None,
                     names=['relation', 'head', 'tail', 'weight'])
    print(f'ConceptNet: {len(df)} triples')
    print(df.head())

    # Build KG tu ConceptNet (subset de test)
    kg_conceptnet = KnowledgeGraph()
    sample_triples = [(row['head'], row['relation'], row['tail'])
                      for _, row in df.head(100000).iterrows()]
    kg_conceptnet.load_triples(sample_triples)

    # Test basic operations
    sample_entity = list(kg_conceptnet.entity2id.keys())[0]
    neighbors = kg_conceptnet.get_neighbors(sample_entity)
    print(f'\nSample entity: {sample_entity}')
    print(f'1-hop neighbors: {len(neighbors)}')
else:
    print(f'ConceptNet not found at {CONCEPTNET_PATH}')
    print('Thay doi CONCEPTNET_PATH cho dung')

ConceptNet: 12397055 triples
                                                      relation  \
/a/[/r/Antonym/,/c/cs/apelativum/n/,/c/en/propr...  /r/Antonym   
/a/[/r/Antonym/,/c/cs/beze/,/c/en/s/]               /r/Antonym   
/a/[/r/Antonym/,/c/cs/extrovert/n/,/c/en/introv...  /r/Antonym   
/a/[/r/Antonym/,/c/cs/introvert/n/,/c/en/extrov...  /r/Antonym   
/a/[/r/Antonym/,/c/cs/kláda/n/,/c/en/hic/]          /r/Antonym   

                                                                  head  \
/a/[/r/Antonym/,/c/cs/apelativum/n/,/c/en/propr...  /c/cs/apelativum/n   
/a/[/r/Antonym/,/c/cs/beze/,/c/en/s/]                       /c/cs/beze   
/a/[/r/Antonym/,/c/cs/extrovert/n/,/c/en/introv...   /c/cs/extrovert/n   
/a/[/r/Antonym/,/c/cs/introvert/n/,/c/en/extrov...   /c/cs/introvert/n   
/a/[/r/Antonym/,/c/cs/kláda/n/,/c/en/hic/]               /c/cs/kláda/n   

                                                               tail  \
/a/[/r/Antonym/,/c/cs/apelativum/n/,/c/en/propr...   /c/en

## 0.5 Build HotpotQA Entity Graph

Xay dung entity graph tu HotpotQA paragraphs bang NER + entity linking

In [9]:
from transformers import pipeline
from collections import defaultdict
import json


def build_hotpotqa_graph(data, max_samples=1000):
    """Xay dung entity graph tu HotpotQA context paragraphs.

    Strategy:
    - Node = title (document) + named entities trong paragraphs
    - Edge = co-occurrence trong cung paragraph + hyperlink
    """
    kg = KnowledgeGraph()
    triples = []

    for sample in data[:max_samples]:
        context = sample['context']
        # Moi context la list cua [title, list_of_sentences]
        titles = [c[0] for c in context]

        # Tao edges giua cac titles co lien quan
        # (supporting facts link cac documents voi nhau)
        for i, (title_i, sents_i) in enumerate(context):
            for j, (title_j, sents_j) in enumerate(context):
                if i < j:
                    # Kiem tra co mention lan nhau khong
                    text_i = ' '.join(sents_i)
                    text_j = ' '.join(sents_j)
                    if title_j.lower() in text_i.lower():
                        triples.append((title_i, 'mentions', title_j))
                    if title_i.lower() in text_j.lower():
                        triples.append((title_j, 'mentions', title_i))

    kg.load_triples(triples)
    return kg


# Build graph tu dev set (nho, de test)
HOTPOTQA_DEV_PATH = os.path.join(HOTPOTQA_DIR, 'hotpot_dev_distractor_v1.json')
if os.path.exists(HOTPOTQA_DEV_PATH):
    with open(HOTPOTQA_DEV_PATH) as f:
        hotpot_dev = json.load(f)

    kg_hotpot = build_hotpotqa_graph(hotpot_dev, max_samples=500)

    # Save
    kg_hotpot.save(os.path.join(KG_DIR, 'hotpotqa_entity_graph.pkl'))
else:
    print('Chay cell download HotpotQA truoc')

Loaded KG: 2141 entities, 1 relations, 1943 triples
Saved KG to /content/drive/MyDrive/ThS/Research/research/COLAB/RUNNING/data/knowledge_graphs/hotpotqa_entity_graph.pkl


## 0.6 Utility Functions (dung chung cho cac phase sau)

Save file nay ra Drive de import trong cac notebook khac

In [ ]:
# === UTILS: Save ra file .py tren Drive de reuse ===

UTILS_DIR = os.path.join(ROOT_DIR, 'utils')
os.makedirs(UTILS_DIR, exist_ok=True)

utils_code = '''
import os
import json
import pickle
import time
import numpy as np
import torch
import networkx as nx
from collections import defaultdict

# === PATHS ===
ROOT_DIR = "/content/drive/MyDrive/ThS/Research/colab/graph_rag"
DATA_DIR = os.path.join(ROOT_DIR, "data")
KG_DIR = os.path.join(DATA_DIR, "knowledge_graphs")
DATASET_DIR = os.path.join(DATA_DIR, "datasets")
CHECKPOINT_DIR = os.path.join(ROOT_DIR, "checkpoints")
RESULTS_DIR = os.path.join(ROOT_DIR, "results")
UTILS_DIR = os.path.join(ROOT_DIR, "utils")


# === KNOWLEDGE GRAPH ===
class KnowledgeGraph:
    """Knowledge Graph wrapper cho research."""

    def __init__(self):
        self.graph = nx.MultiDiGraph()
        self.entity2id = {}
        self.id2entity = {}
        self.relation2id = {}
        self.id2relation = {}
        self.triples = []

    def load_triples(self, triples_list):
        for h, r, t in triples_list:
            for e in [h, t]:
                if e not in self.entity2id:
                    idx = len(self.entity2id)
                    self.entity2id[e] = idx
                    self.id2entity[idx] = e
            if r not in self.relation2id:
                idx = len(self.relation2id)
                self.relation2id[r] = idx
                self.id2relation[idx] = r
            self.triples.append((h, r, t))
            self.graph.add_edge(h, t, relation=r)
        print(f"Loaded KG: {len(self.entity2id)} entities, "
              f"{len(self.relation2id)} relations, {len(self.triples)} triples")

    def load_from_file(self, filepath, sep="\\t"):
        triples = []
        with open(filepath) as f:
            for line in f:
                parts = line.strip().split(sep)
                if len(parts) >= 3:
                    triples.append((parts[0], parts[1], parts[2]))
        self.load_triples(triples)

    def get_neighbors(self, entity, max_hops=1):
        if entity not in self.graph:
            return set()
        if max_hops == 1:
            return set(self.graph.successors(entity)) | set(self.graph.predecessors(entity))
        visited = {entity}
        frontier = {entity}
        for _ in range(max_hops):
            next_frontier = set()
            for node in frontier:
                next_frontier |= set(self.graph.successors(node))
                next_frontier |= set(self.graph.predecessors(node))
            next_frontier -= visited
            visited |= next_frontier
            frontier = next_frontier
        visited.discard(entity)
        return visited

    def extract_subgraph(self, seed_entities, max_hops=2, max_nodes=500):
        nodes = set(seed_entities)
        frontier = set(seed_entities)
        for _ in range(max_hops):
            if len(nodes) >= max_nodes:
                break
            next_frontier = set()
            for node in frontier:
                if node in self.graph:
                    next_frontier |= set(self.graph.successors(node))
                    next_frontier |= set(self.graph.predecessors(node))
            next_frontier -= nodes
            nodes |= next_frontier
            frontier = next_frontier
            if len(nodes) >= max_nodes:
                nodes = set(list(nodes)[:max_nodes])
                break
        return self.graph.subgraph(nodes).copy()

    def shortest_path(self, source, target):
        try:
            return nx.shortest_path(self.graph, source, target)
        except nx.NetworkXNoPath:
            return None

    def to_pyg_data(self):
        from torch_geometric.data import Data
        edge_index = []
        edge_type = []
        for h, r, t in self.triples:
            edge_index.append([self.entity2id[h], self.entity2id[t]])
            edge_type.append(self.relation2id[r])
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_type = torch.tensor(edge_type, dtype=torch.long)
        num_nodes = len(self.entity2id)
        return Data(edge_index=edge_index, edge_type=edge_type, num_nodes=num_nodes)

    def save(self, filepath):
        with open(filepath, "wb") as f:
            pickle.dump(self, f)
        print(f"Saved KG to {filepath}")

    @staticmethod
    def load(filepath):
        with open(filepath, "rb") as f:
            kg = pickle.load(f)
        print(f"Loaded KG: {len(kg.entity2id)} entities, {len(kg.relation2id)} relations")
        return kg


# === EVALUATION METRICS ===
def compute_f1(prediction: str, ground_truth: str) -> float:
    """Token-level F1 score."""
    pred_tokens = prediction.lower().split()
    gt_tokens = ground_truth.lower().split()
    common = set(pred_tokens) & set(gt_tokens)
    if not common:
        return 0.0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gt_tokens)
    return 2 * precision * recall / (precision + recall)


def compute_em(prediction: str, ground_truth: str) -> float:
    """Exact Match score."""
    return float(prediction.strip().lower() == ground_truth.strip().lower())


def compute_hits_at_k(predicted_entities, gold_entities, k=1):
    """Hits@K cho entity prediction."""
    top_k = predicted_entities[:k]
    return float(any(e in gold_entities for e in top_k))


def evaluate_batch(predictions, ground_truths, metrics=["f1", "em"]):
    """Evaluate 1 batch predictions."""
    results = {m: [] for m in metrics}
    for pred, gt in zip(predictions, ground_truths):
        if "f1" in metrics:
            results["f1"].append(compute_f1(pred, gt))
        if "em" in metrics:
            results["em"].append(compute_em(pred, gt))
    return {m: np.mean(v) for m, v in results.items()}


# === VERBALIZE ===
def verbalize_triples(triples):
    """Chuyen KG triples thanh text de dua vao prompt."""
    lines = []
    for h, r, t in triples:
        r_clean = r.replace("_", " ").replace("/", " ")
        lines.append(f"{h} {r_clean} {t}")
    return ". ".join(lines)


def verbalize_paths(paths):
    """Chuyen reasoning paths thanh text."""
    lines = []
    for i, path in enumerate(paths):
        path_str = " -> ".join(path)
        lines.append(f"Path {i+1}: {path_str}")
    return "\\n".join(lines)


# === SAVE/LOAD ===
def save_results(results, name):
    """Save experiment results."""
    filepath = os.path.join(RESULTS_DIR, f"{name}.json")
    with open(filepath, "w") as f:
        json.dump(results, f, indent=2)
    print(f"Results saved to {filepath}")


def load_results(name):
    """Load experiment results."""
    filepath = os.path.join(RESULTS_DIR, f"{name}.json")
    with open(filepath) as f:
        return json.load(f)


def save_checkpoint(model, optimizer, epoch, name):
    """Save training checkpoint."""
    filepath = os.path.join(CHECKPOINT_DIR, f"{name}_ep{epoch}.pt")
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }, filepath)
    print(f"Checkpoint saved to {filepath}")


def load_checkpoint(model, optimizer, filepath):
    """Load training checkpoint."""
    ckpt = torch.load(filepath)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    return ckpt["epoch"]
'''

with open(os.path.join(UTILS_DIR, 'common.py'), 'w') as f:
    f.write(utils_code)

print(f'Utils saved to {UTILS_DIR}/common.py')
print('Trong cac notebook khac, import bang:')
print('  import sys; sys.path.insert(0, "/content/drive/MyDrive/ThS/Research/colab/graph_rag")')
print('  from common import *')

## 0.7 Checklist truoc khi sang Phase 1

- [ ] Thu vien cai dat thanh cong (torch, pyg, transformers, networkx)
- [ ] WebQSP dataset downloaded
- [ ] HotpotQA dataset downloaded
- [ ] ConceptNet accessible tren Drive
- [ ] KnowledgeGraph class hoat dong
- [ ] Utils file saved tren Drive
- [ ] Cau truc thu muc Drive da san sang